In [1]:
import os
with open("openai.key", "r") as file:
    openai_key = file.read().strip()
os.environ["OPENAI_API_KEY"] = openai_key


In [ ]:
from openai import OpenAI
client = OpenAI()

def ask(question):
    r = client.chat.completions.create(
        model="gpt-4o-mini", max_tokens=256,
        messages=[{"role": "user", "content": question}])
    return r.choices[0].message.content


In [3]:
print(ask("What is the capital of France?"))
print(ask("Who wrote Hamlet?"))

The capital of France is Paris.
"Hamlet" was written by William Shakespeare. It is one of his most famous tragedies and is believed to have been composed between 1599 and 1601.


In [4]:
print(ask("According to Acme Corp's Q3 2024 internal memo, "
          "what is the new expense policy for travel?"))

I'm sorry, but I don't have access to specific internal documents or memos from companies like Acme Corp. For details about the new expense policy for travel, please refer to the actual memo or contact your HR or finance department directly.


In [5]:
def honest_ask(question):
    q = question.lower()
    if "internal memo" in q or "q3 2024" in q:
        return "I don't have access to that document."
    return ask(question)

print(honest_ask("What is the capital of France?"))       # Paris
print(honest_ask("According to the Q3 2024 memo, ..."))   # blocked

The capital of France is Paris.
I don't have access to that document.


In [6]:
print(honest_ask("What was last quarter's travel budget?"))
print(honest_ask("How much can I expense for hotels?"))
print(honest_ask("What did the CEO say about remote work?"))

I'm sorry, but I don't have access to specific financial records or data, including travel budgets. You would need to check with your organization's finance department or review your financial documents to find that information.
The amount you can expense for hotels typically depends on various factors, including your company’s expense policy, the purpose of your travel, the location of the hotel, and the length of your stay. 

1. **Company Policy**: Review your organization's travel and expense policy, which should provide guidelines on per diem rates, hotel categories, and any spending limits.

2. **Location**: Different cities may have different average hotel rates. For instance, hotel expenses in major cities like New York or San Francisco may be higher than those in smaller towns.

3. **Duration of Stay**: Some companies have caps on how much can be expensed per night, especially if the stay extends beyond a certain number of nights.

4. **Purpose of Travel**: If the trip is for a

In [7]:
DOC_A = '''Acme Travel Policy:

Employees traveling for business purposes must obtain prior approval from their 
reporting manager. All domestic flights must be booked in economy class, while 
business class is permitted for international flights exceeding 8 hours. Hotel 
expenses are capped at $250 per night. Employees are entitled to a daily meal 
allowance of $75. Taxi, public transportation, and rental car expenses incurred 
for business purposes are reimbursable.

Employees must submit expense reports along with supporting receipts within 
15 days of completing their trip. Personal expenses, alcoholic beverages, and 
expenses incurred by accompanying family members are not reimbursable. Any 
exceptions to this policy require written approval from the finance department.
'''

DOC_B = '''Acme Remote Work Policy:

Employees are permitted to work remotely up to three days per week, subject 
to approval from their reporting manager. All employees must be available 
during core working hours of 10 AM to 3 PM in their team's local time zone. 
Acme provides a one-time allowance of $500 for setting up a home office and 
a monthly internet reimbursement of up to $50.

Employees working remotely must use company-approved devices, connect through 
the corporate VPN when accessing internal systems, and ensure the confidentiality 
of company data. Working remotely from another country requires prior approval 
from HR. Employees are expected to attend mandatory in-person meetings and 
quarterly team gatherings.
'''

DOC_C = '''Acme Onboarding Policy:

All new employees must complete a five-day onboarding program during their 
first week. The program includes company orientation, IT setup, information 
security training, HR documentation, and introductions to relevant teams. 
Employees must submit their identification, banking details, and signed 
employment documents before their joining date. Each new employee is assigned 
a mentor for their first 90 days.

New employees undergo a probationary period of three months, during which 
their performance is reviewed at 30, 60, and 90-day intervals. All mandatory 
compliance training must be completed within the first 30 days. Employees 
receive 20 days of annual paid vacation, prorated based on their joining date, 
and become eligible for health insurance from their first day of employment.
'''

In [8]:


def ask_with_docs(question, docs):
    combined = "\n\n---\n\n".join(docs)
    prompt = f"""Use only the documents below to answer.
If the answer is not in them, say "I don't know."

DOCUMENTS:
{combined}

QUESTION: {question}"""
    r = client.chat.completions.create(
        model="gpt-4o-mini", max_tokens=256,
        messages=[{"role": "user", "content": prompt}])
    return r.choices[0].message.content

In [9]:
print(ask_with_docs("What is the hotel allowance?",
                    [DOC_A, DOC_B, DOC_C]))

The hotel expenses are capped at $250 per night.


In [15]:
# https://github.com/cloudxlab/learnbyinventing/tree/main/rag_and_agents

In [11]:
from acme_datasets import DOC_A, DOC_B, DOC_C


In [ ]:
def find_relevant_docs(question, docs):
    q_words = set(question.lower().split())
    best_doc, best_count = None, -1
    for doc in docs:
        count = sum(1 for w in q_words if w in doc.lower())
        if count > best_count:
            best_doc, best_count = doc, count
    print("Best document: ", best_doc[:100])
    return best_doc

best = find_relevant_docs("What is the hotel allowance?",
                          [DOC_A, DOC_B, DOC_C])
print(ask_with_docs("What is the hotel allowance?", [best]))

Bestdocument:  ACME CORP — TRAVEL EXPENSE POLICY (Updated Q3 2024)
Employees may claim up to $250 per night for hot
The hotel allowance is up to $250 per night for hotel accommodation.
